In [0]:
-- =============================================================================
-- BRIGHT MOTORS CAR SALES ANALYSIS
-- BrightLearn Data Analytics Case Study
-- Data Sets Tables
-- =============================================================================
select * from `workspace`.`default`.`bright_motors_car_sales_data` limit 100;
select * FROM `workspace`.`default`.`bright_motors`limit 100;
select * from `workspace`.`default`.`car_sales_clean_data` limit 100;

-- =============================================================================
-- Platform: Databricks SQL | Spark 3.0 Compatible
-- saledate format: "Tue Dec 16 2014 12:30:00"
-- =============================================================================
-- =============================================================================
-- STEP 1: DATE PARSING HELPER VIEW (Spark 3.0 Safe)
-- Problem:  Spark 3.0 cannot parse "Tue Dec 16 2014 12:30:00" directly
-- Fix:      Strip the 3-letter weekday prefix with REGEXP_REPLACE,
--           then use unix_timestamp() with pattern 'MMM dd yyyy HH:mm:ss'
-- Result:   "Tue Dec 16 2014 12:30:00" → "Dec 16 2014 12:30:00" → DATE
-- =============================================================================
   SELECT
    *,
    -- Remove leading "Tue ", "Mon ", etc. (3 letters + 1 space)
    REGEXP_REPLACE(saledate, '^[A-Za-z]{3}\\s', '')           AS saledate_clean,

    -- unix_timestamp() returns BIGINT (epoch seconds)
    -- Wrap with TIMESTAMP_SECONDS() to get a proper TIMESTAMP, then cast to DATE
    TIMESTAMP_SECONDS(unix_timestamp(REGEXP_REPLACE(saledate, '^[A-Za-z]{3}\\s', ''),
            'MMM dd yyyy HH:mm:ss'  ) )AS sale_timestamp,CAST(TIMESTAMP_SECONDS(unix_timestamp(REGEXP_REPLACE(saledate, '^[A-Za-z]{3}\\s', ''),
                'MMM dd yyyy HH:mm:ss') ) AS DATE ) AS sale_date
    from `workspace`.`default`.`bright_motors_car_sales_data`;
-- =============================================================================
-- STEP 2: CLEAN & TRANSFORMED TABLE
-- =============================================================================
SELECT
    -- Identifiers
    UPPER(TRIM(vin))                                                AS vin,

    -- Time dimensions
    TRY_CAST(year AS INT)                                           AS manufacture_year,
    sale_date,
    YEAR(sale_date)                                                 AS sale_year,
    MONTH(sale_date)                                                AS sale_month,
    QUARTER(sale_date)                                              AS sale_quarter,
    DATE_FORMAT(sale_date, 'MMM')                                   AS sale_month_label,

    -- Vehicle attributes
    INITCAP(TRIM(make))                                             AS make,
    INITCAP(TRIM(model))                                            AS model,
    TRIM(trim)                                                      AS trim_level,
    INITCAP(TRIM(body))                                             AS body_type,
    LOWER(TRIM(transmission))                                       AS transmission,
    UPPER(TRIM(state))                                              AS state,
    INITCAP(TRIM(color))                                            AS exterior_color,
    INITCAP(TRIM(interior))                                         AS interior_color,
    LOWER(TRIM(seller))                                             AS seller,

    -- Numeric fields (NULL-safe)
    COALESCE(TRY_CAST(condition AS DOUBLE), 0)                      AS condition_score,
    COALESCE(TRY_CAST(odometer AS BIGINT), 0)                       AS odometer_miles,
    COALESCE(TRY_CAST(mmr AS DOUBLE), 0)                            AS market_value,
    COALESCE(TRY_CAST(sellingprice AS DOUBLE), 0)                   AS selling_price,

    -- Vehicle age at time of sale
    (YEAR(sale_date) - TRY_CAST(year AS INT))                       AS vehicle_age_years,

    -- Gross profit
    ROUND(
        COALESCE(TRY_CAST(sellingprice AS DOUBLE), 0)
      - COALESCE(TRY_CAST(mmr AS DOUBLE), 0), 2)                    AS gross_profit,

    -- Profit margin %
    ROUND(
        CASE
            WHEN COALESCE(TRY_CAST(sellingprice AS DOUBLE), 0) > 0
            THEN ((COALESCE(TRY_CAST(sellingprice AS DOUBLE), 0)
                 - COALESCE(TRY_CAST(mmr AS DOUBLE), 0))
                 / COALESCE(TRY_CAST(sellingprice AS DOUBLE), 1)) * 100
            ELSE 0
        END, 2
    )                                                               AS profit_margin_pct,

    -- Margin tier
    CASE
        WHEN ((COALESCE(TRY_CAST(sellingprice AS DOUBLE), 0)
             - COALESCE(TRY_CAST(mmr AS DOUBLE), 0))
             / NULLIF(TRY_CAST(sellingprice AS DOUBLE), 0)) * 100 >= 15  THEN 'High Margin'
        WHEN ((COALESCE(TRY_CAST(sellingprice AS DOUBLE), 0)
             - COALESCE(TRY_CAST(mmr AS DOUBLE), 0))
             / NULLIF(TRY_CAST(sellingprice AS DOUBLE), 0)) * 100 >= 5   THEN 'Medium Margin'
        ELSE 'Low Margin'
    END                                                             AS margin_tier,

    -- Mileage bands
    CASE
        WHEN COALESCE(TRY_CAST(odometer AS BIGINT), 0) < 20000     THEN 'Low (<20k)'
        WHEN COALESCE(TRY_CAST(odometer AS BIGINT), 0) < 60000     THEN 'Mid (20k-60k)'
        WHEN COALESCE(TRY_CAST(odometer AS BIGINT), 0) < 100000    THEN 'High (60k-100k)'
        ELSE 'Very High (100k+)'
    END                                                             AS mileage_category,

    -- Condition label
    CASE
        WHEN COALESCE(TRY_CAST(condition AS DOUBLE), 0) >= 4.0     THEN 'Excellent'
        WHEN COALESCE(TRY_CAST(condition AS DOUBLE), 0) >= 3.0     THEN 'Good'
        WHEN COALESCE(TRY_CAST(condition AS DOUBLE), 0) >= 2.0     THEN 'Fair'
        ELSE 'Poor'
    END                                                             AS condition_label

FROM `workspace`.`default`.`bright_motors`
WHERE
    vin                                  IS NOT NULL
    AND sellingprice                     IS NOT NULL
    AND TRY_CAST(sellingprice AS DOUBLE)  > 0
    AND make                             IS NOT NULL
    AND year                             IS NOT NULL
    AND sale_date                        IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY UPPER(TRIM(vin)) ORDER BY sale_date ASC) = 1;


-- Data quality summary
SELECT
    COUNT(*)                                            AS total_records,
    COUNT(DISTINCT vin)                                 AS unique_vehicles,
    COUNT(DISTINCT make)                                AS unique_makes,
    COUNT(DISTINCT state)                               AS unique_states,
    MIN(sale_date)                                      AS earliest_sale,
    MAX(sale_date)                                      AS latest_sale,
    ROUND(AVG(selling_price), 2)                        AS avg_selling_price,
    SUM(CASE WHEN gross_profit < 0 THEN 1 ELSE 0 END)   AS below_market_sales
FROM `workspace`.`default`.`car_sales_clean_data`;


-- =============================================================================
-- STEP 3: ANALYSIS QUERIES
-- =============================================================================

SELECT
    sale_date,
    sale_year,
    sale_month,
    sale_month_label,
    sale_quarter,
    manufacture_year,
    vehicle_age_years,                                                                           ------PRICE vs VEHICLE AGE
    CONCAT('Q', CAST(sale_quarter AS STRING), '-', CAST(sale_year AS STRING)) AS period,         ------QUARTERLY SALES TREND
    make,                                                                                        ------TOP 10 MAKE + MODEL + STATE COMBINATIONS BY REVENUE
    model,
    body_type,                                                                                   ------SALES BY BODY TYPE
    state,
    transmission,                                                                                ------TRANSMISSION TYPE COMPARISON
    mileage_category,                                                                            ------PRICE vs MILEAGE CATEGORY
    condition_label,                                                                             ------CONDITION SCORE IMPACT ON PRICE & MARGIN
    exterior_color,                                                                              ------EXTERIOR COLOR PREFERENCES
    seller,                                                                                      ------TOP 20 SELLERS BY VOLUME
    COUNT(*)                                            AS units_sold,
    ROUND(SUM(selling_price), 2)                       AS total_revenue,
    margin_tier,                                                                                 ------MARGIN TIER DISTRIBUTION
    ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 2)  AS pct_of_total,
    ROUND(AVG(selling_price), 2)                       AS avg_selling_price,
    ROUND(AVG(market_value), 2)                        AS avg_market_value,
    ROUND(SUM(gross_profit), 2)                        AS total_gross_profit,
    ROUND(AVG(odometer_miles), 0)                      AS avg_odometer,
    ROUND(AVG(condition_score), 2)                     AS avg_condition_score,
    ROUND(AVG(profit_margin_pct), 2)                   AS avg_profit_margin_pct,
    ROUND(SUM(selling_price) /
          SUM(SUM(selling_price)) OVER () * 100, 2)    AS revenue_share_pct,                       ------REVENUE & PROFIT BY MAKE
    RANK() OVER (PARTITION BY make ORDER BY SUM(selling_price) DESC) AS rank_within_make,          ------TOP MODELS BY REVENUE (ranked within each make)
    COUNT(DISTINCT make) AS makes_sold,                                                            ------REGIONAL PERFORMANCE BY STATE
     ROUND(
        (SUM(selling_price)
            - LAG(SUM(selling_price)) OVER (ORDER BY sale_year, sale_month))
        / NULLIF(LAG(SUM(selling_price)) OVER (ORDER BY sale_year, sale_month), 0) * 10, 2)AS mom_revenue_growth_pct -----MONTH-OVER-MONTH REVENUE WITH GROWTH RATE
FROM `workspace`.`default`.`car_sales_clean_data`
WHERE sale_year IS NOT NULL
GROUP BY make, model, state, sale_date, sale_year, sale_quarter, sale_month, sale_month_label, body_type,transmission, mileage_category, manufacture_year,
    vehicle_age_years, margin_tier, condition_label, exterior_color, seller
ORDER BY sale_date DESC, sale_year, sale_quarter,total_revenue DESC, units_sold DESC, avg_selling_price DESC, manufacture_year DESC, avg_profit_margin_pct DESC, avg_condition_score DESC;
-------------------------------------------------------------------------------------------------------------------------------------------------------------



 

